In [3]:
# Install packages
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd

In [5]:
data = pd.read_csv("../data/no_missing_animal_id.csv")

data.head()

,animal_id,lactation_number,days_in_milk,reproduction_status,event_date,avg_milk_flow,flow_30_60_session,yield_first_2min_session,yield_session,duration_session_sec,milking
0,-8.839529e+18,1.0,365.0,Pregnant,2019-12-09,2.902991,0.898113,4.975908,11.158372,226,1
1,-5.365332e+18,1.0,66.0,Bred,2021-07-12,3.311224,1.401600,5.347854,15.059267,268,2
2,7.750424e+18,1.0,244.0,Pregnant,2019-12-04,3.900894,4.100475,8.400531,15.331422,231,3
3,5.803650e+18,1.0,394.0,Open,2019-12-14,3.492661,4.399846,8.173735,11.793402,198,1
4,-1.713215e+18,5.0,288.0,Pregnant,2020-05-11,3.311224,4.300056,7.466130,8.572896,153,2


## Sort records from earliest to latest

In [5]:
# Load the cleaned file
no_missing_animal_id = pd.read_csv("../data/no_missing_animal_id.csv")

# Sort records from earliest to latest
no_missing_animal_id["event_date"] = pd.to_datetime(no_missing_animal_id["event_date"])
no_missing_animal_id = no_missing_animal_id.sort_values("event_date").reset_index(drop=True)

# Check earliest and latest records
print("Earliest records:")
print(no_missing_animal_id[["animal_id", "event_date"]].head())

print("Latest records:")
print(no_missing_animal_id[["animal_id", "event_date"]].tail())


Earliest records:
      animal_id event_date
0 -8.344568e+18 2019-06-14
1  7.065605e+18 2019-06-14
2  3.772440e+18 2019-06-14
3 -2.738539e+18 2019-06-14
4 -8.573159e+18 2019-06-14
Latest records:
            animal_id event_date
6650765  2.913070e+18 2021-10-24
6650766 -8.610394e+18 2021-10-24
6650767  7.482694e+18 2021-10-24
6650768 -1.177337e+18 2021-10-24
6650769 -9.898323e+17 2021-10-24


## Splitting sets

In [6]:
# Sort known-ID records from earliest to latest
sorted_data = (
    no_missing_animal_id
    .sort_values("event_date")
    .reset_index(drop=True)
)

# Calculate split points
n = len(sorted_data)
train_end = int(n * 0.70)
validation_end = int(n * 0.80)

# Split the records
train = sorted_data.iloc[:train_end].copy()
validation = sorted_data.iloc[train_end:validation_end].copy()
test = sorted_data.iloc[validation_end:].copy()

# Check sizes and date ranges
for name, subset in [
    ("Training", train),
    ("Validation", validation),
    ("Test", test)
]:
    print(
        name,
        "rows:", len(subset),
        "dates:", subset["event_date"].min(),
        "to", subset["event_date"].max()
    )

# Check the professor's requirement
train_ids = set(train["animal_id"])

print(
    "Validation rows with IDs absent from training:",
    (~validation["animal_id"].isin(train_ids)).sum()
)
print(
    "Test rows with IDs absent from training:",
    (~test["animal_id"].isin(train_ids)).sum()
)

Training rows: 4655539 dates: 2019-06-14 00:00:00 to 2021-03-01 00:00:00
Validation rows: 665077 dates: 2021-03-01 00:00:00 to 2021-05-21 00:00:00
Test rows: 1330154 dates: 2021-05-21 00:00:00 to 2021-10-24 00:00:00
Validation rows with IDs absent from training: 28045
Test rows with IDs absent from training: 232434


## Modeling

In [7]:
import pandas as pd

# Define the target and measurement features
target = "animal_id"

measurement_features = [
    "flow_30_60_session",
    "yield_first_2min_session",
    "yield_session",
    "duration_session_sec",
    "milking",
    "event_date",
]

# Replace event_date with numeric calendar features
def build_features(records):
    X = records[measurement_features].copy()
    dates = pd.to_datetime(X.pop("event_date"), errors="raise")

    X["event_month"] = dates.dt.month
    X["event_day_of_year"] = dates.dt.dayofyear
    X["event_weekday"] = dates.dt.dayofweek
    X["event_year"] = dates.dt.year

    return X.astype("float32")

X_train = build_features(train)
X_valid = build_features(validation)
X_test = build_features(test)

# Treat each animal ID as a category
y_train = train[target].astype("string")
y_valid = validation[target].astype("string")
y_test = test[target].astype("string")

n_cows = y_train.nunique()
n_data_points = len(X_train)


In [9]:
print("Number of training cows:", n_cows)
print("Number of training data points:", n_data_points)
print("Number of features:", X_train.shape[1])
print(X_train.dtypes)

Number of training cows: 7920
Number of training data points: 4655539
Number of features: 9
flow_30_60_session          float32
yield_first_2min_session    float32
yield_session               float32
duration_session_sec        float32
milking                     float32
event_month                 float32
event_day_of_year           float32
event_weekday               float32
event_year                  float32
dtype: object


In [10]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


## Use KNN

In [86]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score


measurement_cols = [
    "flow_30_60_session", "yield_first_2min_session",
    "yield_session", "duration_session_sec", "milking",
    "avg_milk_flow", "days_in_milk", "lactation_number"
]

def build_features(records):
    X = records[measurement_cols].astype("float32").copy()
    dates = pd.to_datetime(records["event_date"], errors="raise")

    # Estimated calving date expressed as days since 2000-01-01
    X["estimated_calving_day"] = (
        (dates - pd.Timestamp("2000-01-01")).dt.days
        - X["days_in_milk"]
    )

    X["event_month"] = dates.dt.month
    X["event_day_of_year"] = dates.dt.dayofyear
    X["event_weekday"] = dates.dt.dayofweek
    X["event_year"] = dates.dt.year
    X["reproduction_status"] = records["reproduction_status"]
    return X

X_train = build_features(train)
X_valid = build_features(validation)
y_train = train["animal_id"].astype(str)
y_valid = validation["animal_id"].astype(str)

numeric_cols = X_train.columns.drop("reproduction_status").tolist()

other_numeric_cols = [
    col for col in numeric_cols
    if col not in ["estimated_calving_day", "lactation_number"]
]

preprocessor = ColumnTransformer(
    [
        ("numeric", StandardScaler(), other_numeric_cols),
        ("calving", StandardScaler(), ["estimated_calving_day"]),
        ("lactation", StandardScaler(), ["lactation_number"]),
        ("status", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ), ["reproduction_status"])
    ],
    transformer_weights={
    "calving": 512.0,
    "lactation": 8.0,
    "status": 0.25
    }
)
knn_model = make_pipeline(
    preprocessor,
    KNeighborsClassifier(
        n_neighbors=7,
        weights="distance",
        metric="minkowski",
        p=1, 
        algorithm="kd_tree",
        n_jobs=1
        )
)

print("Training records:", len(y_train), flush=True)
print("Fitting KNN...", flush=True)
knn_model.fit(X_train, y_train)

print("Predicting validation...", flush=True)
knn_predictions = np.concatenate([
    knn_model.predict(X_valid.iloc[start:start + 128])
    for start in range(0, len(X_valid), 128)
])

print("Validation records:", len(y_valid))
print(f"Accuracy: {accuracy_score(y_valid, knn_predictions):.2%}")
print("Macro F1:", f1_score(
    y_valid, knn_predictions, average="macro", zero_division=0
))

Training records: 4655539
Fitting KNN...
Predicting validation...
Validation records: 665077
Accuracy: 57.71%
Macro F1: 0.4908790105264432


In [80]:
X_test = build_features(test)
y_test = test["animal_id"].astype(str)

training_medians = (
    X_train[numeric_cols]
    .replace([np.inf, -np.inf], np.nan)
    .median()
)

X_test[numeric_cols] = (
    X_test[numeric_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(training_medians)
)

test_predictions = np.concatenate([
    knn_model.predict(X_test.iloc[start:start + 128])
    for start in range(0, len(X_test), 128)
])

print("Test records:", len(y_test))
print(f"Test Accuracy: {accuracy_score(y_test, test_predictions):.2%}")
print("Test Macro F1:", f1_score(
    y_test, test_predictions, average="macro", zero_division=0
))

Test records: 1330154
Test Accuracy: 21.51%
Test Macro F1: 0.18728398454244033


In [81]:
print("Training cows:", train["animal_id"].nunique())
print("Validation cows:", validation["animal_id"].nunique())
print("Testing cows:", test["animal_id"].nunique())

Training cows: 7920
Validation cows: 5174
Testing cows: 5649


In [85]:
print("Total  cows:", no_missing_animal_id["animal_id"].nunique())

Total  cows: 9068


In [88]:
total_records = len(train) + len(validation) + len(test)

print(f"Training records: {len(train):,} ({len(train) / total_records:.2%})")
print(f"Validation records: {len(validation):,} ({len(validation) / total_records:.2%})")
print(f"Test records: {len(test):,} ({len(test) / total_records:.2%})")

Training records: 4,655,539 (70.00%)
Validation records: 665,077 (10.00%)
Test records: 1,330,154 (20.00%)
